# Step 7 — Auxiliary field bound

Bound the linear eigenvalue functional from `eq:eigenvalue-functional-ibp` in the paper's weighted energy norm,

$$
\|u\|_{H^1,\eta_v}^2=\eta_v\|u\|_2^2+\|\nabla u\|_2^2.
$$

A trial auxiliary field w, native interval quadrature, and Step 4/6 data give the bound $M_5^v$ in `eq:dual-riesz-upper-bound`.

Run from `notebooks/` in interval mode. Inputs are the prepared profile, normalized Step 2 eigenfunction, trial `data/external/up_w.mat`, Step 4 odd RHS samples/bounds, Step 6 `step6_odd_matrices_clean.mat`, and the odd inverse trials' `lambda` multipliers. All share the same background, normalized v, retained basis, scale, and inverse trials.

At the eigenpair-ball mass weight `eta_v=0.232` (`eq:45-v-ball-parameters`), the final cell prints `M5_v_upper` and checks it against `14.9512`, which is below the paper bound $M_5^v\le15$ (`eq:45-input-M5v`).

With $D=(I+x\cdot\nabla)/2$, define

$$
H(v)=Dv-\Delta v-(\overline U\cdot\nabla)v+(\nabla\overline U)^Tv.
$$

Then $H(v)=h_0+v/2$, where $h_0$ is the paper's adjoint source. The extra multiple of v vanishes on $v^\perp$. The manuscript inverse fields equal the negatives of the projected raw trials, so the finite-rank source is subtracted:

$$
Z=H(v)-\sum_j\frac{c_j^0}{\nu_j}Q\psi_j-\lambda_wv
+\Delta w_\perp-\nabla p,
$$

$$
M_5^v\le
\left(\|\nabla w_\perp\|_2^2+\frac{\|Z\|_2^2}{\eta_v}\right)^{1/2}
+\delta M_5.
$$

Sections 4–6 construct the fixed source centers $c^0$, their error allowance, and the projected trial. No inverse-field samples or direct source-pairing integrals are recomputed.


## 1. Load fields and Step 4 data

`k=8` is the fixed order supported by native Newton–Cotes, not an arbitrary-order option. The physical grids use 12000 inner radial, 9000 exterior radial and 6000 angular intervals. All satisfy the native panel requirements. No grid enlargement or integration is triggered automatically.

`divfree=true` skips reconstruction and preserves the corrected export's common normalization. The trial uses the original native divergence-free reconstruction. Its velocity and pressure are not rescaled by `eta_v`; its stored `lambda` is included in the sampled field `Z`. Section 4 reads only the inverse multipliers and saved Step 6 matrices to construct fixed source coefficients and a separate error allowance. The pressure operator extracts the weighted pressure gradient from the combined container.



In [1]:
import Pkg
Pkg.activate("..")

include("../src/NS_Numerics.jl")
using .NS_Numerics


  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/3d-navier-stokes-nonuniqueness-main-run`


.Interval{Float64}
Number of threads = 56


In [2]:
# Derivative order and physical grids.
k = 8
padding = derivative_padding(k)
eta_v = dt("0.232")
R = dt(32)
L0, L1 = Pi / dt(2), Pi / dt(2)
Nβ_in, Nβ_out, Nθ = 12000, 9000, 6000

# Background, normalized eigenfunction, and trial field.
UP = from_mat_file("../data/generated/UP_divfree.mat", 2; divfree=true)
up, _ = from_mat_file("../data/generated/up_eig_corrected.mat", 1; divfree=true)
w, lbd_w = from_mat_file("../data/external/up_w.mat", 1)

# Finite-rank data; section 4 reuses the matched Step 6 matrices.
nu = vec(read_matrix("../data/external/eig.mat", ["eig_val_odd"]))
@assert all(x -> !in_interval(0, x), nu)

rhs_bounds = read_matrix("../data/generated/rhs_bound.mat", ["rhs_bound_odd"])

# Frequency ceilings and the inner/exterior domain lengths.
M0, M1 = size(w["u3"].freq_func)
M0_v, M1_v = size(up["u3"].freq_func)
M0_UP, M1_UP = size(UP["u3"].freq_func)

freq_w = dt.([2*M0 + 10, 2*M1 + 10])
freq_v = dt.([2*M0_v + 10, 2*M1_v + 10])
freq_mixed = dt.([2*(M0_UP + M0_v) + 20, 2*(M1_UP + M1_v) + 20])
freq_Z = max.(freq_mixed, freq_w) .+ dt(20)

Beta = atan(R / w.scl_fac)
L_in, L_out = dtype[Beta, L1], dtype[L0 - Beta, L1]

# Padded grids for field samples and derivative bounds.
β_in_pad = grid(Beta, Nβ_in, padding)
β_out_pad = Beta .+ grid(L_out[1], Nβ_out, padding)
θ_pad = grid(L1, Nθ, padding);

## 2. Global source-free derivative priors

Odd class-1 fields have mixed radial parities: use the full $[0,2\pi]$ extension for Bernstein priors. The pressure arrays have the same dimensions as `u3`, so the M-based ceilings cover pressure as well.

Retain five fields: $w$, $v$, $\Delta w$, $\Delta v$ and the initial field
$$Z=H(v)+\Delta w-\nabla p-\lambda_wv.$$
The loaded trial multiplier is fixed and included before refining the bounds for the initial $Z$. The same field and table order is used in both physical regions. The finite-rank source and the projection term $-\gamma\Delta v$ are added later with their own derivative bounds.



In [3]:
W_prior = let
    N0_prior, N1_prior = 12000, 3000
    β = grid(dt(2) * Pi, N0_prior)
    θ = grid(L1, N1_prior)
    domain = dtype[dt(2) * Pi, L1]

    # Reuse the identity and Laplacian samples in both assembled fields and bounds.
    I_mat = get_operator_matrix(w, β, θ, "identity")
    Lap_mat = get_operator_matrix(w, β, θ, "laplacian")
    Iw = apply(I_mat, w)
    Iv = apply(I_mat, up)
    Lw = apply(Lap_mat, w)
    Lv = apply(Lap_mat, up)

    Zfield = mul_A_gradB(up, UP, β, θ; transpose=true) -
        mul_A_gradB(UP, up, β, θ) -
        apply(get_operator_matrix(up, β, θ, "transpose"), up) + Lw -
        apply(get_operator_matrix(w, β, θ, "pressure"), w) - Iv * lbd_w

    # Preserve the table order used by the inner and exterior refinement cells.
    W = []
    for (F, freq) in zip(
        (Iw, Iv, Lw, Lv, Zfield),
        (freq_w, freq_v, freq_w, freq_v, freq_Z),
    )
        push!(W, norm_lst(F, "W$(k+2)Inf"; paras=freq, domain=domain))
    end

    W
end
;


## 3. Inner and exterior samples

### 3.1. Inner region before the finite-rank source is assembled

Retain samples and continuous two-dimensional derivative bounds for the five fields in section 2. The initial $Z$ assembly includes the pressure gradient and the loaded trial multiplier before derivative refinement.

The physical inner grid matches the Step 4 12000×6000 grid: `11:12011` and `11:6011` remove its ten padding layers. The saved samples already contain the radial weight. The matching `rhs_bound_odd` cache supplies both directional derivative bounds for the chosen source. Source bounds are added in section 6 and apply only within $B_{32}$.



In [4]:
(
    Iw_in, Iv_in, Lw_in, Lv_in, Z_in,
    WIw_in, WIv_in, WLw_in, WLv_in, WZ_in,
) = let
    β, θ = β_in_pad, θ_pad

    # Sample the projection fields and Z before choosing the source coefficients.
    I_mat = get_operator_matrix(w, β, θ, "identity")
    Lap_mat = get_operator_matrix(w, β, θ, "laplacian")
    Iw = apply(I_mat, w)
    Iv = apply(I_mat, up)
    Lw = apply(Lap_mat, w)
    Lv = apply(Lap_mat, up)

    Zfield = mul_A_gradB(up, UP, β, θ; transpose=true) -
        mul_A_gradB(UP, up, β, θ) -
        apply(get_operator_matrix(up, β, θ, "transpose"), up) + Lw -
        apply(get_operator_matrix(w, β, θ, "pressure"), w) - Iv * lbd_w

    # Refine the derivative bounds before trimming the padding.
    W = []
    for (F, prior, freq) in zip(
        (Iw, Iv, Lw, Lv, Zfield),
        W_prior,
        (freq_w, freq_v, freq_w, freq_v, freq_Z),
    )
        priorβ = [reshape(prior[m, 1], 3, 1) for m = 1:k+3]
        paras = Linf_to_paras(priorβ, k+2, dtype[0, freq[2]]; ind=1)
        push!(W, norm_lst(
            F, "W$(k)Inf";
            paras=paras, domain=L_in, padding=padding,
        ))
        trim_border!(F, padding)
    end

    (
        Iw, Iv, Lw, Lv, Zfield,
        W[1], W[2], W[3], W[4], W[5],
    )
end
;


### 3.2. Complete exterior region

The outer compactified interval ends at $\pi/2$, hence includes physical infinity. There is no supported Q source here. Fresh native refinement parameters are built for every field and region.

In [5]:
(
    Iw_out, Iv_out, Lw_out, Lv_out, Z_out,
    WIw_out, WIv_out, WLw_out, WLv_out, WZ_out,
) = let
    β, θ = β_out_pad, θ_pad

    # Sample the projection fields and Z before choosing the source coefficients.
    I_mat = get_operator_matrix(w, β, θ, "identity")
    Lap_mat = get_operator_matrix(w, β, θ, "laplacian")
    Iw = apply(I_mat, w)
    Iv = apply(I_mat, up)
    Lw = apply(Lap_mat, w)
    Lv = apply(Lap_mat, up)

    Zfield = mul_A_gradB(up, UP, β, θ; transpose=true) -
        mul_A_gradB(UP, up, β, θ) -
        apply(get_operator_matrix(up, β, θ, "transpose"), up) + Lw -
        apply(get_operator_matrix(w, β, θ, "pressure"), w) - Iv * lbd_w

    # Refine the derivative bounds before trimming the padding.
    W = []
    for (F, prior, freq) in zip(
        (Iw, Iv, Lw, Lv, Zfield),
        W_prior,
        (freq_w, freq_v, freq_w, freq_v, freq_Z),
    )
        priorβ = [reshape(prior[m, 1], 3, 1) for m = 1:k+3]
        paras = Linf_to_paras(priorβ, k+2, dtype[0, freq[2]]; ind=1)
        push!(W, norm_lst(
            F, "W$(k)Inf";
            paras=paras, domain=L_out, padding=padding,
        ))
        trim_border!(F, padding)
    end

    (
        Iw, Iv, Lw, Lv, Zfield,
        W[1], W[2], W[3], W[4], W[5],
    )
end
;


## 4. Source coefficient centers and Step 6 error control

The corrected export is normalized: $\|v\|_2=1$. Set $B_j=Q\psi_j/\nu_j$, $b_j=\langle v,\phi_j\rangle$ and $t_j=\langle B_j,v\rangle$. With pressure gradients omitted in pairings with $v$, the matched residuals satisfy
$$Lv=\lambda v+e,\qquad L\phi_j=\lambda\phi_j+\mu_jv-Q\psi_j+E_j.$$
The native $H(v)$ differs from $L^*v$ by a multiple of $v$, which cancels in the projected coefficient. Hence the exact coefficients obey
$$c_j=\mu_j-\nu_jt_j+\langle v,E_j\rangle-b_j\langle v,e\rangle.$$
Read $\mu_j$ from each inverse file's `lambda` and the remaining quantities from the matched Step 6 cache. Select fixed binary midpoint coefficients $c^0$ from the interval enclosure of $\mu-\nu t$; no inverse-field samples or direct source-pairing integrals are evaluated here.

Let $a_j$ enclose $|(\mu_j-\nu_jt_j)-c_j^0|$. The saved matrices are the complete raw-residual Grams, $(GQ)_{ij}=\langle B_i,B_j\rangle$ and $(GR)_{ij}=\langle E_i,E_j\rangle$. The Hilbert–Schmidt bound gives
$$
\left\|\sum_j B_j\langle v,E_j\rangle\right\|_2
\le\sqrt{\operatorname{tr}(GQ\,GR)}.
$$
Therefore the source error is bounded by
$$
\delta_{L^2}=\sqrt{\operatorname{tr}(GQ\,GR)}
+\sqrt{a^T|GQ|a}
+\varepsilon_v\sqrt{b^TGQb},\qquad
\delta M_5=\frac{\delta_{L^2}}{\sqrt{\eta_v}},\qquad
\varepsilon_v=4.808\times10^{-6}.
$$
All three terms are evaluated with interval arithmetic. The absolute values around the scalar quadratic forms in code preserve upper enclosures when an interval crosses zero; they do not alter the known nonnegativity of the exact norm squares. Only the upper endpoint of the final allowance is used. The whole-space derivative priors and field grids do not enter this coefficient calculation.



In [6]:
using IntervalArithmetic: mid, radius
cs, delta_M5 = let
    GQ, GR, t, b = read_matrix(
        "../data/generated/step6_odd_matrices_clean.mat",
        ["GQ", "GR", "t", "b"],
    )
    t, b = vec(t), vec(b)
    @assert size(GQ) == size(GR) == (length(nu), length(nu))
    @assert length(t) == length(b) == length(nu)

    lbd_phi = zeros(dtype, length(nu))
    for j in eachindex(nu)
        lbd_phi[j] = first(read_matrix(
            "../data/external/phi_odd/up_phi_$(j-1).mat", ["lambda"],
        ))
    end

    cs_base = lbd_phi .- nu .* t
    cs = dt.(mid.(cs_base))
    cs_radius = dt.(radius.(cs_base))

    epsilon_v = dt("4.808e-6")
    source_error = sqrt(abs(sum(GQ .* GR))) +
        sqrt(abs(cs_radius' * abs.(GQ) * cs_radius)) +
        epsilon_v * sqrt(abs(b' * GQ * b))
    delta_M5 = source_error / sqrt(eta_v)

    cs, delta_M5
end
;

## 5. Projection and gradient energy

Native `inner_product` uses continuous two-dimensional Wk tables. The corrected export has $\|v\|_2=1$, so $\gamma=\langle\bar w,v\rangle$ and $w_\perp=\bar w-\gamma v$. The pairing encloses the exact projection coefficient, not its midpoint; no norm of $v$ is recomputed. Integration by parts gives $w_1=\|\nabla w_\perp\|_2^2=-\langle w_\perp,\Delta w_\perp\rangle$ after summing both regions: interface terms cancel, the fields are regular at the origin/axis, and $w,v=O(r^{-2})$, $\nabla w,\nabla v=O(r^{-3})$ remove the boundary flux at infinity.

Retain the full interval projection coefficient in the fields and their derivative bounds. The final energy identity uses only $w_1$, so no trial mass integral is needed.



In [7]:
# Whole-space pairing: sum the inner and exterior contributions.
wv = inner_product(
    Iw_in, Iv_in; W_norms_lst=[WIw_in, WIv_in], domain=L_in,
) + inner_product(
    Iw_out, Iv_out; W_norms_lst=[WIw_out, WIv_out], domain=L_out,
)

# Project the trial field onto v-perp.
γ = wv
γabs = abs(γ)

Wp_in, Wp_out = Iw_in - Iv_in * γ, Iw_out - Iv_out * γ
Lwp_in, Lwp_out = Lw_in - Lv_in * γ, Lw_out - Lv_out * γ
# Continuous derivative bounds for the projected field and its Laplacian.
WWp_in = [
    WIw_in[m, d] .+ γabs .* WIv_in[m, d]
    for m = 1:k+1, d = 1:2
]
WWp_out = [
    WIw_out[m, d] .+ γabs .* WIv_out[m, d]
    for m = 1:k+1, d = 1:2
]
WLwp_in = [
    WLw_in[m, d] .+ γabs .* WLv_in[m, d]
    for m = 1:k+1, d = 1:2
]
WLwp_out = [
    WLw_out[m, d] .+ γabs .* WLv_out[m, d]
    for m = 1:k+1, d = 1:2
]

# Gradient energy from integration by parts over both regions.
w1 = -inner_product(
    Wp_in, Lwp_in; W_norms_lst=[WWp_in, WLwp_in], domain=L_in,
) - inner_product(
    Wp_out, Lwp_out; W_norms_lst=[WWp_out, WLwp_out], domain=L_out,
);


## 6. Source assembly and energy estimate

The target is the same projected functional:
$$
\sup_{\substack{u\ne0,\ \nabla\cdot u=0\\\langle u,v\rangle=0}}
\frac{\left|\left\langle H(v)-\sum_j\frac{c_j}{\nu_j}Q\psi_j,u\right\rangle\right|}
{\sqrt{\eta_v\|u\|_2^2+\|\nabla u\|_2^2}}
\le M_5^v,\qquad \eta_v=0.232.
$$
First estimate the functional with the fixed coefficient centers from section 4. Assemble
$$
q_0=\sum_j\frac{c_j^0}{\nu_j}Q\psi_j,\qquad
Y_0=H(v)-q_0-\lambda_wv,\qquad
Z=Y_0+\Delta w_\perp-\nabla p.
$$
The sampled `Z_in` and `Z_out` initially contain $H(v)+\Delta w-\nabla p-\lambda_wv$. This cell updates them in place by subtracting $\gamma\Delta v$ and $q_0$; $q_0$ is zero in the exterior region. Their derivative bounds `WZ_in` and `WZ_out` are updated by the same terms. Regenerate these samples and bounds before repeating this cell. Mode weights are applied to the loaded RHS in place before summing its mode axis.

**Energy identity.** With $r=Y_0-\eta_vw_\perp+\Delta w_\perp-\nabla p=Z-\eta_vw_\perp$, the same Riesz residual estimate gives
$$
\|\ell_{Y_0}\|_{(H^1,\eta_v)^*}^2
\le 2\langle Y_0,w_\perp\rangle-\eta_v\|w_\perp\|_2^2
-\|\nabla w_\perp\|_2^2+\frac{\|r\|_2^2}{\eta_v}
= w_1+\frac{\|Z\|_2^2}{\eta_v}.
$$
The equality uses $\langle\Delta w_\perp,w_\perp\rangle=-w_1$ and $\langle\nabla p,w_\perp\rangle=0$. Thus the separate source/trial pairing and mass integral are unnecessary. An inaccurate trial velocity, pressure or multiplier is allowed; it changes this computable upper bound.

**Source error.** Section 4 encloses the difference between the exact coefficients and the fixed centers in physical $L^2$. Since $\|u\|_{H^1,\eta_v}\ge\sqrt{\eta_v}\|u\|_2$, the final bound is
$$M_5^v\le\sqrt{w_1+z_2/\eta_v}+\delta M_5,\qquad z_2=\|Z\|_2^2.$$
The code reports only `M5_v_upper`; the center estimate and the source allowance remain available as intermediate variables. The true energy expression is nonnegative. An interval crossing zero at the center square root can therefore still supply an upper endpoint; its decoration remains available through `M5_center`.

**Numerical error control.** The full interval $\gamma$ is propagated through $Z$ and its derivative bounds. The fixed source uses the weighted Step 4 samples on the matching physical grid and the corresponding derivative bounds. Both the gradient energy and $z_2$ are evaluated by native interval `inner_product`, including beta and theta NC8 remainders. Both interval counts must be divisible by six. The compactified exterior reaches infinity; $R=32$ is a splitting point, not a tail cutoff.

The normalized eigenfunction and divergence-free trials give the projection and integration-by-parts identities above. Steps 2, 4, and 6 supply the field, source, and residual bounds used here.



In [8]:
# Assemble the source using the fixed coefficient centers.
Wq = [zeros(dtype, 3) for m = 0:k, d = 1:2]
for j in eachindex(cs), m = 0:k, d = 1:2
    Wq[m+1, d] .+= abs(cs[j] / nu[j]) .* rhs_bounds[:, m+1, d, j]
end
# Step 4 samples already include the radial weight; select the physical grid.
rhs = read_matrix("../data/generated/rhs.mat", ["rhs_odd"])
rhs = rhs[11:12011, 11:6011, :, :]
rhs .*= reshape(cs ./ nu, 1, 1, :, 1)
rhs = sum(rhs; dims=3)
q = VectorFunc(
    w.scl_fac, "space";
    e1=rhs[:, :, 1, 1],
    e2=rhs[:, :, 1, 2],
    e3=rhs[:, :, 1, 3],
)
# Update Z and its bounds in place; regenerate them before repeating this cell.
add_equal!(Z_in, Lv_in * (-γ))
add_equal!(Z_out, Lv_out * (-γ))
minus_equal!(Z_in, q)
WZ_in = [
    WZ_in[m, d] .+ γabs .* WLv_in[m, d] .+ Wq[m, d]
    for m = 1:k+1, d = 1:2
]
WZ_out = [
    WZ_out[m, d] .+ γabs .* WLv_out[m, d]
    for m = 1:k+1, d = 1:2
]

# Final energy estimate and the separate source-error allowance.
z2 = inner_product(
    Z_in, Z_in; W_norms_lst=[WZ_in, WZ_in], domain=L_in,
) + inner_product(
    Z_out, Z_out; W_norms_lst=[WZ_out, WZ_out], domain=L_out,
)
M5_center = sqrt(w1 + z2 / eta_v)
M5_v_upper = sup(M5_center + delta_M5)
println("M5_v_upper = ", M5_v_upper)
@assert M5_v_upper <= inf(dt("14.9512"))


M5_v_upper = 14.951118764198092


## Verification output and the paper's mass weight

At `eta_v=0.232`, the final cell prints

`M5_v_upper = sup(M5_center + delta_M5)`

and asserts `M5_v_upper <= inf(dt("14.9512"))`. Therefore the paper's rounded bound $M_5^v\le15$ in `eq:45-input-M5v` follows from this check. No separate result file is written; `w1`, `z2`, `M5_center`, and `delta_M5` remain available in the kernel.

The mass weight agrees with the paper's eigenpair-ball choice $\eta_v=0.232$ in `eq:45-v-ball-parameters`.

The final assembly updates Z samples and bounds in place. Regenerate them before repeating that cell.
